# Extract Follow Relation from CSV Files

This notebook reads two CSV files (`followers.csv` and `followers_of_followers.csv`) and extracts four fields for each relationship:
- `thread_user_pk`
- `thread_follower_pk`
- `thread_username`
- `thread_follower_username`

Resulting data are concatenated and saved as `{t}_all_followers.csv` where t is in ["AI","ChatGpt", "ML"]


In [13]:
from pathlib import Path
types = ["AI","ChatGpt", "ML"]
DATA_DIR = Path.cwd().parent.parent / 'data' / 'raw' 
DATA_DIR

WindowsPath('c:/Users/pasqua/Desktop/univ/progettoasnm/Code/data_extraction/data/raw')

In [14]:
import pandas as pd

def make_followers_file(df1_path, df2_path, out_path):
    # Legge i due file di input
    df1 = pd.read_csv(df1_path)
    df2 = pd.read_csv(df2_path)

    # -------------------------
    # MAPPAGGIO DF1 (followers)
    # -------------------------

    # --- thread_user_pk per df1 ---
    # Se esiste 'thread_user_pk', lo uso; altrimenti creo una Series di ''
    s1 = df1.get('thread_user_pk', pd.Series([''] * len(df1), index=df1.index)).fillna('')
    # Fallback: user_pk (assume sempre presente in df1)
    user_pk1 = df1['user_pk'].astype(str).fillna('')
    thread_user_pk_1 = s1.where(s1 != '', user_pk1)

    # --- thread_follower_pk per df1 ---
    s2 = df1.get('thread_follower_pk', pd.Series([''] * len(df1), index=df1.index)).fillna('')
    follower_pk1 = df1['follower_pk'].astype(str).fillna('')
    thread_follower_pk_1 = s2.where(s2 != '', follower_pk1)

    # --- thread_username per df1 ---
    s3 = df1.get('thread_username', pd.Series([''] * len(df1), index=df1.index)).fillna('')
    username1 = df1['username'].fillna('')
    thread_username_1 = s3.where(s3 != '', username1).replace('', pd.NA)

    # --- thread_follower_username per df1 ---
    s4 = df1.get('thread_follower_username', pd.Series([''] * len(df1), index=df1.index)).fillna('')
    follower_username1 = df1['follower_username'].fillna('')
    thread_follower_username_1 = s4.where(s4 != '', follower_username1).replace('', pd.NA)

    df1_mapped = pd.DataFrame({
        'thread_user_pk': thread_user_pk_1,
        'thread_follower_pk': thread_follower_pk_1,
        'thread_username': thread_username_1,
        'thread_follower_username': thread_follower_username_1,
        'source': ['followers'] * len(df1)
    })


    # ---------------------------------------
    # MAPPAGGIO DF2 (followers_of_followers)
    # ---------------------------------------

    # --- thread_user_pk per df2: tre livelli di fallback ---
    # 1) 'thread_user_pk'  
    s1 = df2.get('thread_user_pk', pd.Series([''] * len(df2), index=df2.index)).fillna('')
    # 2) 'user_threads_userpk'
    s2 = df2.get('user_threads_userpk', pd.Series([''] * len(df2), index=df2.index)).fillna('')
    # 3) 'user_pk' (assumiamo sempre presente)
    user_pk2 = df2.get('user_pk', pd.Series([''] * len(df2), index=df2.index)).astype(str).fillna('')

    # Applico: se s1 non vuota → s1, altrimenti se s2 non vuota → s2, altrimenti user_pk2
    thread_user_pk_2 = s1.where(s1 != '', s2)
    thread_user_pk_2 = thread_user_pk_2.where(thread_user_pk_2 != '', user_pk2)

    # --- thread_follower_pk per df2: due livelli di fallback ---
    # 1) 'thread_follower_pk'
    s3 = df2.get('thread_follower_pk', pd.Series([''] * len(df2), index=df2.index)).fillna('')
    # 2) 'follower_pk'
    fallback_follower_pk2 = df2.get('follower_pk', pd.Series([''] * len(df2), index=df2.index)).astype(str).fillna('')
    thread_follower_pk_2 = s3.where(s3 != '', fallback_follower_pk2)

    # --- thread_follower_username per df2: fallback su 'follower_username' ---
    s4 = df2.get('thread_follower_username', pd.Series([''] * len(df2), index=df2.index)).fillna('')
    fallback_follower_username2 = df2.get('follower_username', pd.Series([''] * len(df2), index=df2.index)).fillna('')
    thread_follower_username_2 = s4.where(s4 != '', fallback_follower_username2).replace('', pd.NA)

    # --- thread_username per df2: fallback su 'username' ---
    s5 = df2.get('thread_username', pd.Series([''] * len(df2), index=df2.index)).fillna('')
    fallback_username2 = df2.get('username', pd.Series([''] * len(df2), index=df2.index)).fillna('')
    thread_username_2 = s5.where(s5 != '', fallback_username2).replace('', pd.NA)

    df2_mapped = pd.DataFrame({
        'thread_user_pk': thread_user_pk_2,
        'thread_follower_pk': thread_follower_pk_2,
        'thread_username': thread_username_2,
        'thread_follower_username': thread_follower_username_2,
        'source': ['followers_of_followers'] * len(df2)
    })

    # Rimuovo le righe che non hanno né thread_user_pk né thread_follower_pk validi
    mask_valid = (df2_mapped['thread_user_pk'] != '') & (df2_mapped['thread_follower_pk'] != '')
    df2_mapped = df2_mapped.loc[mask_valid].reset_index(drop=True)


    # --------------------
    # CONCAT E PULIZIE FINALI
    # --------------------
    df = pd.concat([df1_mapped, df2_mapped], ignore_index=True)

    # Se i PK sono numeri in formato float, rimuovo .0 finale
    df["thread_user_pk"] = df["thread_user_pk"].apply(
        lambda x: str(float(x)).split('.')[0] if pd.notna(x) and x != '' else x
    )
    df["thread_follower_pk"] = df["thread_follower_pk"].apply(
        lambda x: str(float(x)).split('.')[0] if pd.notna(x) and x != '' else x
    )

    # Mostro un’anteprima e salvo
    print(df.head())
    df.to_csv(out_path, index=False)
    print(f'Saved extracted CSV to {out_path}')


In [15]:
for t in types:
    PATH_FOLLOWERS = DATA_DIR / t / 'followers.csv'
    PATH_FOLLOWERS_OF_FOLLOWERS = DATA_DIR / t / 'followers_of_followers.csv'
    PATH_INTERIM = DATA_DIR.parent / 'interim' /  f'{t.lower()}_all_followers.csv'
    make_followers_file(PATH_FOLLOWERS, PATH_FOLLOWERS_OF_FOLLOWERS, PATH_INTERIM)

C:\Users\pasqua\AppData\Local\Temp\ipykernel_9952\3323236072.py:6: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(df2_path)


  thread_user_pk thread_follower_pk thread_username thread_follower_username  \
0    63310777769        67036773813        mlssfshn              stacidblack   
1    63310777769         2384555054        mlssfshn                leotw4552   
2    63310777769        63465676707        mlssfshn                homeecmel   
3    63310777769        63400405482        mlssfshn       handmade_market_ks   
4    63310777769        69029504312        mlssfshn              l0dservices   

      source  
0  followers  
1  followers  
2  followers  
3  followers  
4  followers  
Saved extracted CSV to c:\Users\pasqua\Desktop\univ\progettoasnm\Code\data_extraction\data\interim\ai_all_followers.csv
  thread_user_pk thread_follower_pk thread_username thread_follower_username  \
0    63073067781        71347280359  aliencoremuzik               arts.wh0re   
1    63073067781        65191160590  aliencoremuzik                cc.jayyyy   
2    63073067781        70294896056  aliencoremuzik           aliquas